# Week 13: Import a Public Dataset into Atlas

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lolusername/CST4714_DB_admin/blob/main/week_13/materials/notebooks/06_public_data_capacity_integration.ipynb)

[View on GitHub](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_13/materials/notebooks/06_public_data_capacity_integration.ipynb)

**One 100-minute Thanksgiving-week class. Work individually.** Save your own copy in Drive before editing. The required database is **real MongoDB Atlas**, using the retained 75-record CISA teaching snapshot. A Colab runtime runs Python; PyMongo sends requests; Atlas stores the documents. No mock replaces a failed connection.

**Follow this route:** Cells 01, 02a–02b, 03–08, 10, 11 **twice**, 14 and 15. Skip optional Cells 09, 12 and 13. Run one cell at a time; Cell 10 pauses while you add the runtime IP rule. Do not use Run all for the initial setup.

The instructor guides the connection and first vendor query. The **20-minute lab** starts afterward: change Cell 15 from vendor to product, check the output, and write 5–7 sentences in Submission Record. Submit **one viewable Colab sharing link**, with your saved outputs and explanation, in Brightspace. No separate dataset report, screenshot or file attachment.

Use an existing Free cluster and your own database user, prepared before class. This notebook needs no paid tier, backup subscription, or sharded deployment. SQLite and PostgreSQL remain optional practice paths below; they are not evidence of an Atlas connection. The live feed is optional. Cell 07 is a guided range/hash simulation; it does not deploy shards.


## Data reference before the first query

One record is one historical CISA Known Exploited Vulnerabilities catalog entry, not a computer or victim. A CVE ID names a vulnerability entry; it does not count affected machines. The required snapshot has **75 records and eight selected fields**. [Open the complete record reference and field dictionary](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_13/DATA_REFERENCE.md).

A document stores named fields and their values. The Atlas collection groups these documents. We preserve seven text fields, including canonical `YYYY-MM-DD` calendar-day strings, and the actual `cwes` list. We do not invent a time of day or silently convert a day into an instant. SQL naming/conversions are optional reference. Missing source fields become Python `None` and must pass the checks before any write. The retained sample is not current vulnerability-management advice.


## Resource and License Boundary

The notebook prose and code are course OER. The CISA records come from an official
U.S. government feed and are not represented as original course data. The
embedded snapshot preserves source metadata and a description of the field
selection. It is a compact classroom fixture, not a current vulnerability-
management source.

Official feed:
<https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json>

In [ ]:
#@title Cell 01 — Import standard-library helpers
from bisect import bisect_right
from collections import Counter
from datetime import date
from getpass import getpass
from hashlib import sha256
from ipaddress import IPv4Address
import json
import re
import sqlite3
import urllib.request
from uuid import uuid4

print("Python's built-in libraries are ready. No cloud connection was opened.")

## 1. Inspect the source and its records

Keep `USE_LIVE_FEED = False` for this lesson. The embedded historical sample makes the class outputs reproducible; it does not make Atlas work offline. Run the supplied collapsed Cell 02b after the visible source-choice cell. Its metadata records the original source version and retrieval date. Do not replace that date with today's lesson date.

The live-feed option is optional later exploration. Its records and grouped results can differ from this historical snapshot.


In [ ]:
#@title Cell 02a — Source choice (keep visible)
USE_LIVE_FEED = False


In [ ]:
#@title Cell 02b — Load the retained historical snapshot or selected feed
OFFLINE_SNAPSHOT = json.loads(r'''{"title": "CISA Catalog of Known Exploited Vulnerabilities", "catalogVersion": "2026.07.10", "sourceDateReleased": "2026-07-10T17:00:25.7327Z", "retrievedAt": "2026-07-13T06:19:55.069283+00:00", "sourceUrl": "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json", "selection": "first 75 source records; documented fields only", "count": 75, "vulnerabilities": [{"cveID": "CVE-2026-56291", "vendorProject": "Balbooa", "product": "Forms", "vulnerabilityName": "Balbooa Forms Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-10", "dueDate": "2026-07-13", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-48939", "vendorProject": "iCagenda", "product": "iCagenda", "vulnerabilityName": "iCagenda Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-10", "dueDate": "2026-07-13", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-48908", "vendorProject": "JoomShaper", "product": "SP Page Builder", "vulnerabilityName": "JoomShaper SP Page Builder Unrestricted Upload of File with Dangerous Type Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-434"]}, {"cveID": "CVE-2026-55255", "vendorProject": "Langflow", "product": "Langflow", "vulnerabilityName": "Langflow Authorization Bypass Through User-Controlled Key Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-639"]}, {"cveID": "CVE-2026-56290", "vendorProject": "Joomlack", "product": "Page Builder", "vulnerabilityName": "Joomlack Page Builder Improper Access Control Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-48282", "vendorProject": "Adobe", "product": "ColdFusion", "vulnerabilityName": "Adobe ColdFusion Path Traversal Vulnerability", "dateAdded": "2026-07-07", "dueDate": "2026-07-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-45659", "vendorProject": "Microsoft", "product": "SharePoint Server", "vulnerabilityName": "Microsoft SharePoint Server Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-07-01", "dueDate": "2026-07-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-502"]}, {"cveID": "CVE-2026-48558", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Authentication Bypass Vulnerability", "dateAdded": "2026-06-29", "dueDate": "2026-07-02", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-347"]}, {"cveID": "CVE-2026-12569", "vendorProject": "PTC", "product": "Windchill and FlexPLM", "vulnerabilityName": "PTC Windchill and FlexPLM Improper Input Validation Vulnerability", "dateAdded": "2026-06-25", "dueDate": "2026-06-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20", "CWE-502"]}, {"cveID": "CVE-2026-20230", "vendorProject": "Cisco", "product": "Unified Communications Manager", "vulnerabilityName": "Cisco Unified Communications Manager Server-Side Request Forgery (SSRF) Vulnerability", "dateAdded": "2026-06-25", "dueDate": "2026-06-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-918"]}, {"cveID": "CVE-2025-67038", "vendorProject": "Lantronix", "product": "EDS5000", "vulnerabilityName": "Lantronix EDS5000 Code Injection Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78", "CWE-94"]}, {"cveID": "CVE-2026-34910", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Improper Input Validation Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2026-34909", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Path Traversal Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-34908", "vendorProject": "Ubiquiti", "product": "UniFi OS", "vulnerabilityName": "Ubiquiti UniFi OS Improper Access Control Vulnerability", "dateAdded": "2026-06-23", "dueDate": "2026-06-26", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-20253", "vendorProject": "Splunk", "product": "Enterprise", "vulnerabilityName": "Splunk Enterprise Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-06-18", "dueDate": "2026-06-21", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-48907", "vendorProject": "Widget Factory", "product": "Joomla Content Editor ", "vulnerabilityName": "Widget Factory Joomla Content Editor Improper Access Control Vulnerability", "dateAdded": "2026-06-16", "dueDate": "2026-06-19", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-284"]}, {"cveID": "CVE-2026-54420", "vendorProject": "LiteSpeed", "product": "cPanel Plugin", "vulnerabilityName": "LiteSpeed cPanel Plugin UNIX Symbolic Link (Symlink) Following Vulnerability", "dateAdded": "2026-06-15", "dueDate": "2026-06-18", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-61"]}, {"cveID": "CVE-2026-20262", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Directory or Path Traversal Vulnerability", "dateAdded": "2026-06-15", "dueDate": "2026-06-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-35273", "vendorProject": "Oracle", "product": " PeopleSoft Enterprise PeopleTools", "vulnerabilityName": "Oracle PeopleSoft Enterprise PeopleTools Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-06-12", "dueDate": "2026-06-15", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-10520", "vendorProject": "Ivanti", "product": "Sentry", "vulnerabilityName": "Ivanti Sentry OS Command Injection Vulnerability", "dateAdded": "2026-06-11", "dueDate": "2026-06-14", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78"]}, {"cveID": "CVE-2026-11645", "vendorProject": "Google", "product": "Chromium V8", "vulnerabilityName": "Google Chromium V8 Out-of-Bounds Read and Write Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-787", "CWE-125"]}, {"cveID": "CVE-2026-7473", "vendorProject": "Arista", "product": "Extensible Operating System", "vulnerabilityName": "Arista Extensible Operating System Incomplete Comparison with Missing Factors Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-1023"]}, {"cveID": "CVE-2026-20245", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Improper Encoding or Escaping of Output Vulnerability", "dateAdded": "2026-06-09", "dueDate": "2026-06-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-116"]}, {"cveID": "CVE-2026-42271", "vendorProject": "BerriAI", "product": "LiteLLM", "vulnerabilityName": "BerriAI LiteLLM Command Injection Vulnerability", "dateAdded": "2026-06-08", "dueDate": "2026-06-22", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-78", "CWE-77"]}, {"cveID": "CVE-2026-50751", "vendorProject": "Check Point", "product": "Security Gateway", "vulnerabilityName": "Check Point Security Gateway Improper Authentication Vulnerability", "dateAdded": "2026-06-08", "dueDate": "2026-06-11", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-287"]}, {"cveID": "CVE-2026-28318", "vendorProject": "SolarWinds", "product": "Serv-U", "vulnerabilityName": "SolarWinds Serv-U Uncontrolled Resource Consumption Vulnerability", "dateAdded": "2026-06-05", "dueDate": "2026-06-19", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-400"]}, {"cveID": "CVE-2026-45247", "vendorProject": "Mirasvit", "product": "Mirasvit Full Page Cache Warmer", "vulnerabilityName": "Mirasvit Full Page Cache Warmer Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-06-03", "dueDate": "2026-06-06", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-502"]}, {"cveID": "CVE-2022-0492", "vendorProject": "Linux", "product": "Kernel", "vulnerabilityName": "Linux Kernel Improper Authentication Vulnerability", "dateAdded": "2026-06-02", "dueDate": "2026-06-05", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287", "CWE-862"]}, {"cveID": "CVE-2025-48595", "vendorProject": "Android", "product": "Framework", "vulnerabilityName": "Android Framework Integer Overflow Vulnerability", "dateAdded": "2026-06-02", "dueDate": "2026-06-05", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-190"]}, {"cveID": "CVE-2024-21182", "vendorProject": "Oracle", "product": "WebLogic Server", "vulnerabilityName": "Oracle WebLogic Server Unspecified Vulnerability", "dateAdded": "2026-06-01", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2026-0257", "vendorProject": "Palo Alto Networks", "product": "PAN-OS", "vulnerabilityName": "Palo Alto Networks PAN-OS Authentication Bypass Vulnerability", "dateAdded": "2026-05-29", "dueDate": "2026-06-01", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-565"]}, {"cveID": "CVE-2026-48027", "vendorProject": "Nx", "product": "Nx Console", "vulnerabilityName": "Nx Console Embedded Malicious Code Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-06-10", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-506"]}, {"cveID": "CVE-2026-45321", "vendorProject": "TanStack", "product": "TanStack", "vulnerabilityName": "TanStack Unspecified Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-06-10", "knownRansomwareCampaignUse": "Known", "cwes": []}, {"cveID": "CVE-2026-8398", "vendorProject": "Daemon", "product": "Daemon Tools Lite", "vulnerabilityName": "Daemon Tools Lite Embedded Malicious Code Vulnerability", "dateAdded": "2026-05-27", "dueDate": "2026-05-30", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-506"]}, {"cveID": "CVE-2026-48172", "vendorProject": "LiteSpeed", "product": "cPanel Plugin", "vulnerabilityName": "LiteSpeed cPanel Plugin Privilege Escalation Vulnerability", "dateAdded": "2026-05-26", "dueDate": "2026-05-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-266"]}, {"cveID": "CVE-2026-9082", "vendorProject": "Drupal", "product": "Core", "vulnerabilityName": "Drupal Core SQL Injection Vulnerability", "dateAdded": "2026-05-22", "dueDate": "2026-05-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-89"]}, {"cveID": "CVE-2025-34291", "vendorProject": "Langflow", "product": "Langflow", "vulnerabilityName": "Langflow Origin Validation Error Vulnerability", "dateAdded": "2026-05-21", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-346"]}, {"cveID": "CVE-2026-34926", "vendorProject": "Trend Micro", "product": "Apex One", "vulnerabilityName": "Trend Micro Apex One (On-Premise) Directory Traversal Vulnerability", "dateAdded": "2026-05-21", "dueDate": "2026-06-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-23"]}, {"cveID": "CVE-2008-4250", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Buffer Overflow Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-94"]}, {"cveID": "CVE-2009-1537", "vendorProject": "Microsoft", "product": "DirectX", "vulnerabilityName": "Microsoft DirectX NULL Byte Overwrite Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2009-3459", "vendorProject": "Adobe", "product": "Acrobat and Reader", "vulnerabilityName": "Adobe Acrobat and Reader Heap-Based Buffer Overflow Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-119"]}, {"cveID": "CVE-2010-0249", "vendorProject": "Microsoft", "product": "Internet Explorer", "vulnerabilityName": "Microsoft Internet Explorer Use-After-Free Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-416"]}, {"cveID": "CVE-2010-0806", "vendorProject": "Microsoft", "product": "Internet Explorer", "vulnerabilityName": "Microsoft Internet Explorer Use-After-Free Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-399"]}, {"cveID": "CVE-2026-41091", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Link Following Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-59"]}, {"cveID": "CVE-2026-45498", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Denial of Service Vulnerability", "dateAdded": "2026-05-20", "dueDate": "2026-06-03", "knownRansomwareCampaignUse": "Unknown", "cwes": []}, {"cveID": "CVE-2026-42897", "vendorProject": "Microsoft", "product": "Microsoft", "vulnerabilityName": "Microsoft Exchange Server Cross-Site Scripting Vulnerability", "dateAdded": "2026-05-15", "dueDate": "2026-05-29", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-79"]}, {"cveID": "CVE-2026-20182", "vendorProject": "Cisco", "product": "Catalyst SD-WAN", "vulnerabilityName": "Cisco Catalyst SD-WAN Controller Authentication Bypass Vulnerability", "dateAdded": "2026-05-14", "dueDate": "2026-05-17", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287"]}, {"cveID": "CVE-2026-42208", "vendorProject": "BerriAI", "product": "LiteLLM", "vulnerabilityName": "BerriAI LiteLLM SQL Injection Vulnerability", "dateAdded": "2026-05-08", "dueDate": "2026-05-11", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-89"]}, {"cveID": "CVE-2026-6973", "vendorProject": "Ivanti", "product": "Endpoint Manager Mobile (EPMM)", "vulnerabilityName": "Ivanti Endpoint Manager Mobile (EPMM) Improper Input Validation Vulnerability", "dateAdded": "2026-05-07", "dueDate": "2026-05-10", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2026-0300", "vendorProject": "Palo Alto Networks", "product": "PAN-OS", "vulnerabilityName": "Palo Alto Networks PAN-OS Out-of-bounds Write Vulnerability", "dateAdded": "2026-05-06", "dueDate": "2026-05-09", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-787"]}, {"cveID": "CVE-2026-31431", "vendorProject": "Linux", "product": "Kernel", "vulnerabilityName": "Linux Kernel Incorrect Resource Transfer Between Spheres Vulnerability", "dateAdded": "2026-05-01", "dueDate": "2026-05-15", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-669"]}, {"cveID": "CVE-2026-41940", "vendorProject": "WebPros", "product": "cPanel & WHM and WP2 (WordPress Squared)", "vulnerabilityName": "WebPros cPanel & WHM and WP2 (WordPress Squared) Missing Authentication for Critical Function Vulnerability", "dateAdded": "2026-04-30", "dueDate": "2026-05-03", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-306"]}, {"cveID": "CVE-2024-1708", "vendorProject": "ConnectWise", "product": "ScreenConnect", "vulnerabilityName": "ConnectWise ScreenConnect Path Traversal Vulnerability", "dateAdded": "2026-04-28", "dueDate": "2026-05-12", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-22"]}, {"cveID": "CVE-2026-32202", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Protection Mechanism Failure Vulnerability", "dateAdded": "2026-04-28", "dueDate": "2026-05-12", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-693"]}, {"cveID": "CVE-2025-29635", "vendorProject": "D-Link", "product": "DIR-823X", "vulnerabilityName": "D-Link DIR-823X Command Injection Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-77"]}, {"cveID": "CVE-2024-7399", "vendorProject": "Samsung", "product": "MagicINFO 9 Server", "vulnerabilityName": "Samsung MagicINFO 9 Server Path Traversal Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22", "CWE-434"]}, {"cveID": "CVE-2024-57728", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Path Traversal Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-22"]}, {"cveID": "CVE-2024-57726", "vendorProject": "SimpleHelp ", "product": "SimpleHelp", "vulnerabilityName": "SimpleHelp Missing Authorization Vulnerability", "dateAdded": "2026-04-24", "dueDate": "2026-05-08", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-862"]}, {"cveID": "CVE-2026-39987", "vendorProject": "Marimo", "product": "Marimo", "vulnerabilityName": "Marimo Remote Code Execution Vulnerability", "dateAdded": "2026-04-23", "dueDate": "2026-05-07", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-306"]}, {"cveID": "CVE-2026-33825", "vendorProject": "Microsoft", "product": "Defender", "vulnerabilityName": "Microsoft Defender Insufficient Granularity of Access Control Vulnerability", "dateAdded": "2026-04-22", "dueDate": "2026-05-06", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-1220"]}, {"cveID": "CVE-2026-20122", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manger", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Incorrect Use of Privileged APIs Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-648"]}, {"cveID": "CVE-2026-20133", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Exposure of Sensitive Information to an Unauthorized Actor Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-200"]}, {"cveID": "CVE-2025-2749", "vendorProject": "Kentico", "product": "Kentico Xperience", "vulnerabilityName": "Kentico Xperience Path Traversal Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-22", "CWE-434"]}, {"cveID": "CVE-2023-27351", "vendorProject": "PaperCut", "product": "NG/MF", "vulnerabilityName": "PaperCut NG/MF Improper Authentication Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-287"]}, {"cveID": "CVE-2025-48700", "vendorProject": "Synacor", "product": "Zimbra Collaboration Suite (ZCS)", "vulnerabilityName": "Synacor Zimbra Collaboration Suite (ZCS) Cross-site Scripting Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-79"]}, {"cveID": "CVE-2026-20128", "vendorProject": "Cisco", "product": "Catalyst SD-WAN Manager", "vulnerabilityName": "Cisco Catalyst SD-WAN Manager Storing Passwords in a Recoverable Format Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-04-23", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-257"]}, {"cveID": "CVE-2025-32975", "vendorProject": "Quest", "product": "KACE Systems Management Appliance (SMA)", "vulnerabilityName": "Quest KACE Systems Management Appliance (SMA) Improper Authentication Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-287"]}, {"cveID": "CVE-2024-27199", "vendorProject": "JetBrains", "product": "TeamCity", "vulnerabilityName": "JetBrains TeamCity Relative Path Traversal Vulnerability", "dateAdded": "2026-04-20", "dueDate": "2026-05-04", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-23"]}, {"cveID": "CVE-2026-34197", "vendorProject": "Apache", "product": "ActiveMQ", "vulnerabilityName": "Apache ActiveMQ Improper Input Validation Vulnerability", "dateAdded": "2026-04-16", "dueDate": "2026-04-30", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20", "CWE-94"]}, {"cveID": "CVE-2009-0238", "vendorProject": "Microsoft", "product": "Office", "vulnerabilityName": "Microsoft Office Remote Code Execution", "dateAdded": "2026-04-14", "dueDate": "2026-04-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-94"]}, {"cveID": "CVE-2026-32201", "vendorProject": "Microsoft", "product": "SharePoint Server", "vulnerabilityName": "Microsoft SharePoint Server Improper Input Validation Vulnerability", "dateAdded": "2026-04-14", "dueDate": "2026-04-28", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-20"]}, {"cveID": "CVE-2012-1854", "vendorProject": "Microsoft", "product": "Visual Basic for Applications (VBA)", "vulnerabilityName": "Microsoft Visual Basic for Applications Insecure Library Loading Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-426"]}, {"cveID": "CVE-2025-60710", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Link Following Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-59"]}, {"cveID": "CVE-2023-21529", "vendorProject": "Microsoft", "product": "Exchange Server", "vulnerabilityName": "Microsoft Exchange Server Deserialization of Untrusted Data Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Known", "cwes": ["CWE-502"]}, {"cveID": "CVE-2023-36424", "vendorProject": "Microsoft", "product": "Windows", "vulnerabilityName": "Microsoft Windows Out-of-Bounds Read Vulnerability", "dateAdded": "2026-04-13", "dueDate": "2026-04-27", "knownRansomwareCampaignUse": "Unknown", "cwes": ["CWE-125"]}]}''')

OFFICIAL_FEED = "https://www.cisa.gov/sites/default/files/feeds/known_exploited_vulnerabilities.json"

if USE_LIVE_FEED:
    try:
        with urllib.request.urlopen(OFFICIAL_FEED, timeout=30) as response:
            source_package = json.load(response)
    except (OSError, ValueError):
        raise RuntimeError("Feed download failed. Set USE_LIVE_FEED = False to use the teaching snapshot.") from None
    source_mode = "current official feed"
else:
    source_package = OFFLINE_SNAPSHOT
    source_mode = "embedded versioned teaching snapshot"

if not isinstance(source_package, dict):
    raise ValueError("Expected a catalog object. No database writes have run.")

print("Source mode:", source_mode)
print("Catalog version:", source_package.get("catalogVersion") or source_package.get("sourceCatalogVersion"))
print("Source release time:", source_package.get("dateReleased") or source_package.get("sourceDateReleased"))
print("Retrieval time:", source_package.get("retrievedAt", "live request in this runtime"))
print("Source URL:", source_package.get("sourceUrl", OFFICIAL_FEED))

### Normalize Only the Fields Used by This Lesson

The live feed contains more fields than the teaching snapshot. We intentionally
select the same compact fields in both paths. This is a modeling decision, not a
claim that omitted fields are unimportant.

In [ ]:
#@title Cell 03 — Normalize the eight lesson fields
raw_records = source_package.get("vulnerabilities")
if not isinstance(raw_records, list) or not raw_records:
    raise ValueError("Expected a nonempty vulnerabilities list. Stop before loading.")
if not all(isinstance(raw, dict) for raw in raw_records[:75]):
    raise ValueError("Each selected vulnerability must be a JSON object.")

# Keep the lesson small even when the current feed is selected.
records = []
for raw in raw_records[:75]:
    records.append({
        "cveID": raw.get("cveID"),
        "vendorProject": raw.get("vendorProject"),
        "product": raw.get("product"),
        "vulnerabilityName": raw.get("vulnerabilityName"),
        "dateAdded": raw.get("dateAdded"),
        "dueDate": raw.get("dueDate"),
        "knownRansomwareCampaignUse": raw.get("knownRansomwareCampaignUse"),
        "cwes": raw.get("cwes"),
    })

print("Selected records:", len(records))
print("Selected fields:", list(records[0]))
print("Example record:\n", json.dumps(records[0], indent=2))

### Check the records before a database write

A successful JSON parse only means Python could read the text. Cell 04 checks required text, CVE format, duplicate IDs, real calendar dates and a list of CWE strings. A **unique key** identifies one logical record; repeating the same CVE should update that entry rather than create another entry.

Dates describe calendar days. The required Atlas path preserves canonical `YYYY-MM-DD` strings, as does SQLite; optional PostgreSQL uses its `date` type and converts it back for comparison. The `cwes` field remains a real array in Atlas. An empty list does not prove there is no weakness. We preserve labels exactly, including the trailing space in `SimpleHelp `, so a hidden cleanup cannot change a grouping.


In [ ]:
#@title Cell 04 — Check IDs, dates, nulls, and duplicates
fields = list(records[0])
required_text = fields[:-1]  # All selected fields except the CWE list.
quality_errors = []
whitespace_labels = []
for record in records:
    for field in required_text:
        value = record[field]
        if not isinstance(value, str) or not value.strip():
            quality_errors.append((field, "expected nonempty text"))
    if not re.fullmatch(r"CVE-[0-9]{4}-[0-9]{4,}", str(record["cveID"])):
        quality_errors.append(("cveID", "expected a CVE identifier"))
    for field in ("dateAdded", "dueDate"):
        try:
            parsed_date = date.fromisoformat(record[field])
            if parsed_date.isoformat() != record[field]:
                quality_errors.append((field, "use YYYY-MM-DD"))
        except (TypeError, ValueError):
            quality_errors.append((field, "invalid calendar date"))
    if not isinstance(record["cwes"], list) or not all(
        isinstance(value, str) and value.strip() for value in record["cwes"]
    ):
        quality_errors.append(("cwes", "expected a list of nonempty strings"))
    vendor = record["vendorProject"]
    if isinstance(vendor, str) and vendor != vendor.strip():
        whitespace_labels.append(repr(vendor))

if quality_errors:
    raise ValueError(f"Fix the source before loading: {quality_errors[:5]}")
cve_counts = Counter(record["cveID"] for record in records)
duplicate_ids = sorted(key for key, count in cve_counts.items() if count > 1)
if duplicate_ids:
    raise ValueError("Duplicate CVE identifiers. Stop and decide which record is authoritative.")
print("Passed required-text, ID, date, list, and uniqueness checks:", len(records))
print("Vendor labels with preserved leading/trailing spaces:", sorted(set(whitespace_labels)))

### Ask a Question Before Choosing a Database Shape

Our first question is: **Which vendors occur most often in this selected
snapshot?** This describes the sample, not all vulnerabilities and not a vendor's
security quality. The limited sample and source order matter.

In [ ]:
#@title Cell 05 — Inspect a vendor grouping
vendor_counts = Counter(record["vendorProject"] for record in records)
print("Five most frequent vendors in this selected sample:")
for vendor, count in sorted(vendor_counts.items(), key=lambda item: (-item[1], item[0]))[:5]:
    print(f"  {vendor!r}: {count}")

## 2. Measure distribution before choosing infrastructure

Cell 06 supplies two useful measurements. **Cardinality** is the number of distinct values. **Largest value share** is the fraction of the sample using its most common value. Compare `cveID` with `dateAdded`: an ID can identify one record while a repeated date cannot.

These measurements do not justify sharding 75 records. **Sharding** splits data across servers; **replication** keeps copies for availability. A good import key is not automatically a good shard key or a way to target vendor queries. Keep the full four-candidate output for reference; focus on CVE ID and date added today.


In [ ]:
#@title Cell 06 — Measure candidate key distributions
candidate_values = {
    "vendorProject": [record["vendorProject"] for record in records],
    "dateAdded": [record["dateAdded"] for record in records],
    "cveID": [record["cveID"] for record in records],
    "vendorProject + product": [
        (record["vendorProject"], record["product"]) for record in records
    ],
}

print(f"{'candidate':28} {'distinct':>8} {'ratio':>8} {'largest value share':>20}")
for name, values in candidate_values.items():
    frequencies = Counter(values)
    distinct = len(frequencies)
    largest_share = max(frequencies.values()) / len(values)
    print(f"{name:28} {distinct:8d} {distinct / len(values):8.2f} {largest_share:20.2%}")

dates_in_source_order = [record["dateAdded"] for record in records]
descending_date_order = all(
    left >= right for left, right in zip(dates_in_source_order, dates_in_source_order[1:])
)
print("\nDate-added values are monotonic descending in source order:", descending_date_order)
print("This describes input order, not production write order.")

### Guided range and hashed placement

**Run supplied Cell 07 with the instructor.** It is a deterministic placement simulation, not a real sharded MongoDB deployment or a performance test. The oldest 80 percent sets date boundaries; the newest 20 percent acts like later writes. SHA-256 maps CVE IDs into four teaching buckets. MongoDB uses different hashing and balancing behavior. Observe where the 15 later records land. No separate response is required.


In [ ]:
#@title Cell 07 — Compare range and hash placement
chronological = sorted(records, key=lambda record: record["dateAdded"])
split_at = max(1, int(len(chronological) * 0.80))
historical = chronological[:split_at]
later_writes = chronological[split_at:]

historical_dates = sorted(record["dateAdded"] for record in historical)
range_boundaries = [
    historical_dates[int(len(historical_dates) * fraction)]
    for fraction in (0.25, 0.50, 0.75)
]
range_buckets = Counter(
    bisect_right(range_boundaries, record["dateAdded"])
    for record in later_writes
)
hash_buckets = Counter(
    int(sha256(record["cveID"].encode("utf-8")).hexdigest(), 16) % 4
    for record in later_writes
)

print("Historical date boundaries:", range_boundaries)
print("Later-write range buckets:", dict(sorted(range_buckets.items())))
print("Later-write teaching hash buckets:", dict(sorted(hash_buckets.items())))
print("Later records tested:", len(later_writes))
print("These fixed teaching buckets do not model MongoDB range migrations or latency.")
assert sum(range_buckets.values()) == len(later_writes)
assert sum(hash_buckets.values()) == len(later_writes)

### Interpret the measurements without another assignment

A CVE ID identifies one entry. A date may identify several entries. Neither fact alone tells us whether production needs more capacity. Consider data size, latency and throughput under a measured workload before adding infrastructure. The 75-record fixture is too small to establish that need.

Use one measured comparison in the short final handoff. No separate sharding matrix, discussion activity or capacity report is required.


## 3. Connect to Atlas, import twice, and verify

Leave `TARGET = "atlas"` for the required lesson. Cell 08 chooses **one run's** names; do not rerun it between imports. The fixed database is `cst4714_week13`; the collection is a new `kev_<run suffix>`. Scope the database user's `readWrite` permission to this database. Separate names prevent accidental resets, not access between users who share permissions.

An **upsert** replaces a matching record or inserts one if its key is absent. An **idempotent** import can be repeated with the same source without adding duplicate logical records. We use `cveID` plus a unique index. The import updates source IDs; it does not delete records removed from a future feed, retain previous versions, or provide one transaction over the entire batch.

SQLite and PostgreSQL are optional alternatives, not the required class route. To explore a different target later, retain your evidence, run the optional cleanup, then start this section with that target.


In [ ]:
#@title Cell 08 — Choose Atlas and create isolated practice names
TARGET = "atlas"  # Required today. Optional later: "sqlite" or "postgres".
if TARGET not in {"sqlite", "atlas", "postgres"}:
    raise ValueError("Choose sqlite, atlas, or postgres.")
if globals().get("practice_active", False):
    raise RuntimeError("This run is still open. Repeat its IMPORT cell, or retain evidence and use cleanup before starting another run.")

run_id = uuid4().hex[:12]
atlas_database_name = "cst4714_week13"
atlas_collection_name = "kev_" + run_id
postgres_schema = "cst4714_data_" + run_id
atlas_client = None
atlas_collection = None
atlas_import_history = []
postgres_connection = None
con = None
rows = [
    (record["cveID"], record["vendorProject"], record["product"],
     record["vulnerabilityName"], record["dateAdded"], record["dueDate"],
     record["knownRansomwareCampaignUse"], json.dumps(record["cwes"]))
    for record in records
]
practice_active = True
print("Selected target:", TARGET)
if TARGET == "atlas":
    print("Database:", atlas_database_name)
    print("This run's collection:", atlas_collection_name)
    print("Names chosen only. Connect in Cell 10, then import with Cell 11.")
if TARGET == "sqlite":
    con = sqlite3.connect(":memory:")
    con.execute('''
        CREATE TABLE kev_sample (
            cve_id TEXT PRIMARY KEY, vendor_project TEXT NOT NULL,
            product TEXT NOT NULL, vulnerability_name TEXT NOT NULL,
            date_added TEXT NOT NULL, due_date TEXT NOT NULL,
            ransomware_use TEXT, cwes_json TEXT NOT NULL
        )
    ''')


### Optional SQLite import — skip for the Atlas lesson

Cell 09 runs only if you deliberately selected `TARGET = "sqlite"`. It is local Python SQLite, not Atlas. The transaction covers that SQL batch. Repeat it without resetting the in-memory database. This optional path adds no required assignment.


### Trace the records before this experiment

![Record relationships for Week 13](https://raw.githubusercontent.com/lolusername/CST4714_DB_admin/main/week_13/illustrations/lesson_relationships.png)

The [complete data reference](https://github.com/lolusername/CST4714_DB_admin/blob/main/week_13/DATA_REFERENCE.md) provides the same relationships as readable text.


In [ ]:
#@title Cell 09 — Optional SQLite import and repeat
if TARGET == "sqlite":
    before_count = con.execute("SELECT count(*) FROM kev_sample").fetchone()[0]
    with con:
        con.executemany('''
        INSERT INTO kev_sample VALUES (?, ?, ?, ?, ?, ?, ?, ?)
        ON CONFLICT (cve_id) DO UPDATE SET
            vendor_project = excluded.vendor_project,
            product = excluded.product,
            vulnerability_name = excluded.vulnerability_name,
            date_added = excluded.date_added,
            due_date = excluded.due_date,
            ransomware_use = excluded.ransomware_use,
            cwes_json = excluded.cwes_json
        ''', rows)
    after_count = con.execute("SELECT count(*) FROM kev_sample").fetchone()[0]
    print("SQLite records before / after:", before_count, after_count)


### Required Atlas connection — prepare the account before class

Open your own **Free** cluster. Use a **database user**, distinct from the Atlas website login, with `readWrite` scoped to `cst4714_week13`. Do not choose a paid upgrade or enable a backup subscription. You need your database URI, not an Atlas API key. [Official driver connection instructions](https://www.mongodb.com/docs/atlas/driver-connection/).

Cell 10 prints the public IPv4 of the Colab runtime. Add that exact address with **`/32`** as a temporary project IP access-list entry, with an appropriate expiry. `/32` allows one IPv4 address. The browser's Add Current IP may select your laptop instead. Do not open `0.0.0.0/0`. After the rule is active, return and press Enter. [Official IP access-list instructions](https://www.mongodb.com/docs/atlas/security/ip-access-list/).

Copy the Python connection URI from **Connect > Drivers**, completing its database credentials privately. Reserved password characters need percent-encoding. Keep it in the optional Colab secret `MONGODB_URI` with notebook access, or paste into the hidden prompt. Never put it in a code cell or screenshot. TLS and certificate checks remain on. The cell pings and creates an empty uniquely named collection's index. Ping proves a server response; Cell 11's writes and Cell 14's read-back provide data evidence. No mock fallback is used.

The IP lookup and driver download need internet. A restarted runtime may get another IP. Failure messages omit the credential-bearing driver error; a failed setup closes the client so Cell 10 can be retried. Atlas UI navigation here is documentation-based, not proof that your account has already connected.


In [ ]:
#@title Cell 10 — Connect to real Atlas with a hidden URI
if TARGET == "atlas" and atlas_client is None:
    %pip -q install pymongo
    from urllib.parse import urlsplit
    from pymongo import MongoClient, ReadPreference
    from pymongo.server_api import ServerApi
    from pymongo.read_concern import ReadConcern
    from pymongo.write_concern import WriteConcern

    try:
        with urllib.request.urlopen("https://api4.ipify.org", timeout=10) as response:
            runtime_ip = str(IPv4Address(response.read().decode().strip()))
    except Exception:
        raise RuntimeError("Could not obtain the runtime IPv4. Retry; do not open access from everywhere.") from None
    print("Temporary Atlas IP access-list entry:", runtime_ip + "/32")
    input("After adding this runtime /32 rule in Atlas, press Enter: ")
    atlas_uri = None
    try:
        try:
            from google.colab import userdata
            atlas_uri = userdata.get("MONGODB_URI")
        except Exception:
            atlas_uri = getpass("Atlas Drivers URI (hidden): ")
        host = (urlsplit(atlas_uri).hostname or "").lower()
        if not atlas_uri.startswith("mongodb+srv://") or not host.endswith(".mongodb.net"):
            raise ValueError("Use the standard Atlas Drivers URI.")
        if "<" in atlas_uri or ">" in atlas_uri:
            raise ValueError("Complete the credential placeholders privately.")
        atlas_client = MongoClient(
            atlas_uri, tls=True, tlsInsecure=False,
            server_api=ServerApi("1", strict=True, deprecation_errors=True),
            serverSelectionTimeoutMS=10000, timeoutMS=10000,
            appname="CST4714-Week13-Classroom",
        )
        atlas_client.admin.command("ping")
        print("Atlas responded to ping. No source records have been imported yet.")
        atlas_collection = atlas_client[atlas_database_name].get_collection(
            atlas_collection_name, write_concern=WriteConcern(w="majority"),
            read_preference=ReadPreference.PRIMARY, read_concern=ReadConcern("majority"))
        assert atlas_collection.count_documents({}) == 0, "Expected this run's fresh collection."
        atlas_collection.create_index("cveID", unique=True, name="unique_cve_id")
        print("Ready to import. Database:", atlas_database_name)
        print("Collection:", atlas_collection_name)
        print("Initial record count:", atlas_collection.count_documents({}))
    except Exception:
        if atlas_client is not None:
            atlas_client.close()
        atlas_client = None
        atlas_collection = None
        raise RuntimeError(
            "Atlas connection or collection setup failed. Check the standard Drivers URI, "
            "database user/password, readWrite on cst4714_week13, runtime /32, DNS and TLS. "
            "Keep certificate checks enabled. No mock was substituted."
        ) from None
    finally:
        atlas_uri = None
elif TARGET == "atlas":
    print("Keep this connected run. Repeat Cell 11; do not regenerate collection names.")


### Import into the same Atlas collection twice

Run **Cell 11 twice without rerunning Cell 08 or changing the source**. A stable CVE ID plus `replace_one(..., upsert=True)` selects one logical entry. The first pass inserts absent IDs; the second matches existing IDs. Expected count path: **0 → 75 → 75**. With unchanged source, the second pass matches 75 IDs and inserts none. The driver's modified counter can still be nonzero even when the selected stored values are equal; it is not our proof of equality. Our local test observed this without establishing the server-internal cause. Cell 14 checks the actual selected values independently.

The unique index prevents duplicate `cveID` values. Each replacement is atomic for one document; this 75-document loop is **not** one all-or-nothing transaction. If interrupted, the target can contain a partial batch. Rerun the same checked source into that target and verify it afterward. [Official replacement/upsert behavior](https://www.mongodb.com/docs/languages/python/pymongo-driver/current/crud/replace/).

Keep the printed collection name. In Atlas Data Explorer, select the correct project and cluster, database **cst4714_week13**, and that exact **kev_...** collection. In Documents, filter `{"cveID": "CVE-2026-56291"}` and click Find to inspect the same example. Do not create a second collection manually.


In [ ]:
#@title Cell 11 — Import into Atlas and repeat without resetting
if TARGET == "atlas":
    from copy import deepcopy
    if atlas_collection is None:
        raise RuntimeError("Complete Cell 10 successfully before importing.")
    before_count = atlas_collection.count_documents({})
    inserted_count = matched_count = modified_count = 0
    for record in records:
        result = atlas_collection.replace_one(
            {"cveID": record["cveID"]}, deepcopy(record), upsert=True)
        inserted_count += int(result.upserted_id is not None)
        matched_count += result.matched_count
        modified_count += result.modified_count
    after_count = atlas_collection.count_documents({})
    atlas_import_history.append((before_count, after_count))
    print("Atlas records before / after:", before_count, after_count)
    print("Inserted / matched / modified:", inserted_count, matched_count, modified_count)
    print("Completed import count pairs:", atlas_import_history)
    assert after_count == len(records), "Count differs. Read and verify the target before continuing."


### Optional Supabase/PostgreSQL Path

Run this path only when `TARGET = "postgres"`. In Supabase's **Connect** panel,
select the **Session pooler** connection string. Its shared pooler supports IPv4,
including notebook networks that cannot reach a direct IPv6 endpoint. Use the
database password and the exact pooler username from the panel, not the Supabase
website password or API key. Enter the URL only in the hidden prompt.

If the copied URI ends in `/postgres`, append `?sslmode=require`. If it already
has query parameters after a `?`, append `&sslmode=require`. If `sslmode` is
already present, update that value instead of adding it twice. Keep
`verify-ca` or `verify-full` if you have configured the required certificate.
URL-encode reserved characters in the password only, once (for example,
`@` becomes `%40`); do not encode the whole connection URI.

The URI must request `sslmode=require`, `verify-ca`, or `verify-full`. `require`
provides encrypted transport but does not guarantee server identity verification.
Use `verify-full` with the provider's CA certificate for certificate and hostname
verification. This notebook does not disable TLS to solve a connection problem.

The next cell creates a uniquely named practice schema. `sql.Identifier` quotes
that schema name correctly; `%s` placeholders are for **values**, not table names.
A quoted identifier and a value parameter solve different problems.
Autocommit keeps the connection out of an idle transaction between cells. Each
`transaction()` block still commits its complete operation on success or rolls it
back on an error. `SET LOCAL search_path` applies only inside that transaction.

In [ ]:
#@title Cell 12 — Optional PostgreSQL connection
if TARGET == "postgres" and postgres_connection is None:
    %pip -q install "psycopg[binary]"
    import psycopg
    from psycopg import sql
    from psycopg.conninfo import conninfo_to_dict

    postgres_url = getpass("PostgreSQL connection URL (hidden): ")
    try:
        sslmode = conninfo_to_dict(postgres_url).get("sslmode", "prefer")
    except psycopg.Error:
        postgres_url = None
        raise ValueError("The connection URL could not be parsed. Copy a fresh Session pooler URI.") from None
    if sslmode not in {"require", "verify-ca", "verify-full"}:
        postgres_url = None
        raise ValueError(
            "Encrypted transport is required. Append ?sslmode=require to a URI "
            "without query parameters, or &sslmode=require after existing parameters. "
            "If sslmode already exists, change its value instead of adding it twice."
        )
    try:
        postgres_connection = psycopg.connect(
            postgres_url, connect_timeout=10, autocommit=True, prepare_threshold=None
        )
        with postgres_connection.transaction():
            with postgres_connection.cursor() as cursor:
                cursor.execute("SET LOCAL statement_timeout = '10s'")
                cursor.execute(sql.SQL("CREATE SCHEMA {}").format(sql.Identifier(postgres_schema)))
                cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                    sql.Identifier(postgres_schema)))
                cursor.execute('''
                CREATE TABLE kev_sample (
                    cve_id text PRIMARY KEY,
                    vendor_project text NOT NULL,
                    product text NOT NULL,
                    vulnerability_name text NOT NULL,
                    date_added date NOT NULL,
                    due_date date NOT NULL,
                    ransomware_use text,
                    cwes jsonb NOT NULL
                )
                ''')
    except Exception:
        if postgres_connection is not None:
            postgres_connection.close()
        postgres_connection = None
        raise RuntimeError(
            "PostgreSQL connection/setup failed. Check the session-pooler URL, "
            "database password, project status, permissions, and sslmode=require "
            "or verify-full with its CA certificate."
        ) from None
    finally:
        postgres_url = None
    print("PostgreSQL is connected. Practice schema:", postgres_schema)

### Import Into the Existing PostgreSQL Table

Run this import cell twice. The transaction covers the complete batch. If one row
fails, this transaction rolls back instead of leaving a half-imported batch.
`excluded` names the proposed row that collided with the primary key. `jsonb`
preserves the CWE list instead of squeezing multiple values into one delimited
text label. SQL values travel separately from the SQL command.

In [ ]:
#@title Cell 13 — Optional PostgreSQL import
if TARGET == "postgres":
    with postgres_connection.transaction():
        with postgres_connection.cursor() as cursor:
            cursor.execute("SET LOCAL statement_timeout = '10s'")
            cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                sql.Identifier(postgres_schema)))
            cursor.execute("SELECT count(*) FROM kev_sample")
            before_count = cursor.fetchone()[0]
            cursor.executemany('''
                INSERT INTO kev_sample VALUES
                    (%s, %s, %s, %s, %s, %s, %s, %s::jsonb)
                ON CONFLICT (cve_id) DO UPDATE SET
                    vendor_project = excluded.vendor_project,
                    product = excluded.product,
                    vulnerability_name = excluded.vulnerability_name,
                    date_added = excluded.date_added,
                    due_date = excluded.due_date,
                    ransomware_use = excluded.ransomware_use,
                    cwes = excluded.cwes
            ''', rows)
            cursor.execute("SELECT count(*) FROM kev_sample")
            after_count = cursor.fetchone()[0]
    print("PostgreSQL records before / after:", before_count, after_count)

### Verify all values and types, not just a count

Cell 14 checks the count, every selected field of the known first record, and—in the required Atlas path—**all 75 documents against the checked source**. It independently reads the database, excluding only MongoDB's generated `_id`. It also checks that the seven text fields are strings and `cwes` is a list of strings. A wrong label or a date converted to another type must fail even if the count remains 75.

These checks establish this retained import's result. They do not establish that the subset is the full live catalog, that every other user has correct permissions, that backups work, or that production performance is adequate.


In [ ]:
#@title Cell 14 — Read back and compare source values and types
expected_known = records[0]
known_id = expected_known["cveID"]

if TARGET == "sqlite":
    observed_count = con.execute("SELECT count(*) FROM kev_sample").fetchone()[0]
    known_row = con.execute(
        "SELECT * FROM kev_sample WHERE cve_id = ?",
        [known_id],
    ).fetchone()
    assert known_row is not None, "The selected source ID is missing."
    observed_known = dict(zip(fields, known_row))
    observed_known["cwes"] = json.loads(observed_known["cwes"])

if TARGET == "atlas":
    observed_count = atlas_collection.count_documents({})
    stored_records = list(atlas_collection.find({}, {"_id": 0}).sort("cveID", 1))
    observed_known = atlas_collection.find_one({"cveID": known_id}, {"_id": 0})
    expected_records = sorted(records, key=lambda record: record["cveID"])
    assert stored_records == expected_records, "At least one stored value differs from the source."
    assert all(
        all(isinstance(record[field], str) for field in fields if field != "cwes")
        and isinstance(record["cwes"], list)
        and all(isinstance(item, str) for item in record["cwes"])
        for record in stored_records
    ), "A stored field has the wrong type."
    print("All 75 source records and selected field types match." if not USE_LIVE_FEED
          else "All selected source records and field types match.")

if TARGET == "postgres":
    with postgres_connection.transaction():
        with postgres_connection.cursor() as cursor:
            cursor.execute("SET LOCAL statement_timeout = '10s'")
            cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                sql.Identifier(postgres_schema)))
            cursor.execute("SELECT count(*) FROM kev_sample")
            observed_count = cursor.fetchone()[0]
            cursor.execute(
                "SELECT * FROM kev_sample WHERE cve_id = %s",
                (known_id,),
            )
            known_row = cursor.fetchone()
    assert known_row is not None, "The selected source ID is missing."
    observed_known = dict(zip(fields, known_row))
    for field in ("dateAdded", "dueDate"):
        observed_known[field] = observed_known[field].isoformat()

print("Expected / observed records:", len(records), observed_count)
print("Known source record:", expected_known)
print("Stored record:", observed_known)
assert observed_count == len(records), "Record count differs from the checked source."
assert observed_known == expected_known, "A stored value differs, even if the count is correct."
print("The selected record matches all eight source fields.")


### Your 20-minute lab: change vendor groups to product groups

The instructor first runs Cell 15 with `GROUP_FIELD = "vendorProject"`. **Change that one line to `GROUP_FIELD = "product"` and rerun the same cell.** A group combines records with the same selected label. `$sum: 1` counts one per catalog entry. `$sort` puts larger counts first and breaks ties alphabetically; `$limit: 5` displays only the first five groups.

The last lines independently compute the same groups directly from the source using Python's `Counter`. The assertion compares the ordered database result with that source result. Explain what one product row means: entries in this historical subset, not affected computers or the security quality of a vendor. `GROUP_FIELD` is the field name; `"$" + GROUP_FIELD` tells MongoDB to read that field's value.

The two allowed field names also keep optional SQL queries bounded. Values and column names are different things; those paths retain parameterized value lookups. You do not need to run them today.


In [ ]:
#@title Cell 15 — Your change: group by product
GROUP_FIELD = "vendorProject"  # Student change: "product".
allowed_columns = {"vendorProject": "vendor_project", "product": "product"}
if GROUP_FIELD not in allowed_columns:
    raise ValueError("Choose vendorProject or product.")
group_column = allowed_columns[GROUP_FIELD]

if TARGET == "sqlite":
    grouped = con.execute(f'''
        SELECT {group_column}, count(*) AS vulnerability_count
        FROM kev_sample GROUP BY {group_column}
        ORDER BY vulnerability_count DESC, {group_column} COLLATE BINARY
        LIMIT 5
    ''').fetchall()

if TARGET == "atlas":
    result = list(atlas_collection.aggregate([
        {"$group": {"_id": "$" + GROUP_FIELD, "n": {"$sum": 1}}},
        {"$sort": {"n": -1, "_id": 1}},
        {"$limit": 5},
    ]))
    grouped = [(row["_id"], row["n"]) for row in result]

if TARGET == "postgres":
    with postgres_connection.transaction():
        with postgres_connection.cursor() as cursor:
            cursor.execute("SET LOCAL statement_timeout = '10s'")
            cursor.execute(sql.SQL("SET LOCAL search_path TO {}").format(
                sql.Identifier(postgres_schema)))
            cursor.execute(sql.SQL('''
                SELECT {field}, count(*) AS vulnerability_count
                FROM kev_sample GROUP BY {field}
                ORDER BY vulnerability_count DESC, {field} COLLATE "C"
                LIMIT 5
            ''').format(field=sql.Identifier(group_column)))
            grouped = cursor.fetchall()

source_groups = Counter(record[GROUP_FIELD] for record in records)
expected_groups = sorted(source_groups.items(), key=lambda item: (-item[1], item[0]))[:5]
print("Grouped field:", GROUP_FIELD)
for label, count in grouped:
    print(f"{label!r}: {count}")
assert grouped == expected_groups, "Database and source groupings disagree."
print("The five displayed groups match the checked source.")

## Submission Record — write 5–7 sentences here

Use your own saved evidence to explain: (1) the historical source and one measured CVE-ID/date comparison; (2) what the two imports did to the same collection; (3) one row in your changed product result and an independent verification; and (4) one limit, such as the small historical subset or lack of a production performance test.

Write in this text cell. This is a brief handoff to a future maintainer, not a separate report or partner activity. If you only observed the instructor's Atlas demonstration, label that observation honestly; do not claim your own connection succeeded. Finish the same notebook under the existing Brightspace deadline. No extra assignment replaces a setup failure.


## Optional final-project transfer and cleanup

The existing final-project assignment remains unchanged. This lesson can supply a pattern for source provenance, stable IDs and verification, but no project response or additional deliverable is required here.

Keep `RUN_CLEANUP = False` until the outputs, changed query and handoff are saved and you have inspected Atlas. Optionally set it to True afterward. The cleanup removes only this run's uniquely named Atlas collection (or the deliberately selected optional SQL practice target), then closes the connection. It does not drop the whole Atlas database. Remove the temporary runtime `/32` rule through the project's normal process when no longer needed. Closing a Python client alone does not remove that network entry.

After cleanup, Cell 08 can create a new isolated run. To repeat the import during the lesson, use **Cell 11**, not Cell 08.


## Submit one saved Colab link

Save the changed Cell 15, both import outputs, checks and 5–7 sentence handoff in your Drive copy. Choose Share, give the instructor viewer access, and submit **that copy's Colab URL in Brightspace**. Use the existing Brightspace due date. Do not submit the GitHub starter link, an `.ipynb` attachment, a screenshot, or another report. Check that no credential appears in saved code or output. The instructor does not need your database password to review the saved evidence.


In [ ]:
#@title Cell 16 — Optional cleanup after retaining your evidence
RUN_CLEANUP = False  # Set True only after both query runs and your handoff.

if not RUN_CLEANUP:
    print("Cleanup skipped: finish both group queries and the handoff first.")
else:
    if atlas_client is not None:
        try:
            atlas_client[atlas_database_name].drop_collection(atlas_collection_name)
            print("Removed this run's Atlas collection. Other collections were kept.")
        finally:
            atlas_client.close()
            atlas_client = None
            atlas_collection = None
        print("Remove the temporary runtime /32 rule in Atlas after class.")
    if postgres_connection is not None:
        try:
            with postgres_connection.cursor() as cursor:
                cursor.execute(sql.SQL("DROP TABLE IF EXISTS {}.kev_sample").format(
                    sql.Identifier(postgres_schema)))
                try:
                    cursor.execute(sql.SQL("DROP SCHEMA IF EXISTS {}").format(
                        sql.Identifier(postgres_schema)))
                except psycopg.errors.DependentObjectsStillExist:
                    print("Kept the practice schema because it contains other objects.")
            print("Removed this run's PostgreSQL table. No CASCADE was used.")
        finally:
            postgres_connection.close()
            postgres_connection = None
    if con is not None:
        con.close()
        con = None
        print("Closed the disposable SQLite database.")
    practice_active = False
